## Denoising Diffusion Probabilistic Models

In [ ]:
import os
import statistics

import torch
import torchvision
from torchvision.transforms.functional import to_pil_image
from cleanfid import fid
from IPython.display import display

import torchutils

from models import DDPM
from utils.gen_utils import gen

In [ ]:
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

dtype = torchutils.get_torch_dtype("bf16")

In [ ]:
def load_checkpoint(checkpoint_path, device="cuda"):
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    model = DDPM(
		img_size=checkpoint["img_size"],
		in_channels=checkpoint["in_channels"],
		timesteps=checkpoint["timesteps"],
	)
    model.load_state_dict(checkpoint["ddpm"])
    model = model.to(device)
    model.eval()
    return model

### AFHQ

In [ ]:
afhq_checkpoint_path = os.path.join("checkpoints", "afhq_500000.pth")

In [ ]:
afhq_model = load_checkpoint(afhq_checkpoint_path, device=device)

In [ ]:
afhq_fid_scores = []
for _ in range(10):
	afhq_fid_score = fid.compute_fid(
		gen=lambda z: gen(afhq_model, z, dtype, device),
		dataset_name="afhq",
		dataset_res=64,
		num_gen=10_000,
		dataset_split="custom"
	)
	afhq_fid_scores.append(afhq_fid_score)

In [ ]:
afhq_fid_mean = statistics.mean(afhq_fid_scores)
print(f"Mean: {afhq_fid_mean}")

In [ ]:
afhq_fid_stdev = statistics.stdev(afhq_fid_scores)
print(f"Standard Deviation: {afhq_fid_stdev}")

In [ ]:
fake_images = afhq_model.generate_samples(num_samples=9*9, dtype=dtype)

grid = torchvision.utils.make_grid(
    fake_images.detach().cpu(),
    nrow=9,
    normalize=False,
	value_range=(-1, 1)
)

pil_image = to_pil_image(grid)
display(pil_image)